# Блок 4 — дообучение эмбеддера re-ID (Этап B, Colab GPU)

Магистерская ВКР «Прикладной ИИ» (ТГУ). Cross-fit ArcFace-finetune эмбеддера на узоре брюшка тритонов
(TK + PW). **KPI:** top-1 ≥ 75 % и top-5 ≥ 95 % на kpi_core.

Ноутбук **тонкий**: вся логика — в протестированном пакете `triton_crop`. Здесь только запуск на GPU +
выгрузка результатов в Drive. Два эмбеддера переключаются одной строкой `MODEL` в ячейке 1.

**Как запускать:**
1. Локально: `python -m triton_crop.cli embed-pack` → залей `triton_block4_pack.zip` в `MyDrive/triton_block4/`.
2. `Среда выполнения → Сменить среду → T4 GPU`.
3. Ячейка 1: выбери `MODEL = 'mega'` или `'miewid'`. → `Выполнить все`.

**Правила (уроки triton 2.0):**
- `EPOCHS` (ячейка 1) **не менять между resume** одного прогона (cosine T = epochs — расписание LR поедет).
- Сброс рантайма ~90 мин — **норма**: снова `Выполнить все`, resume подхватит готовые фолды из Drive.
- Всё пишется в `RUN_DIR` (Drive) — переживает сброс. Mega и MiewID пишут в РАЗНЫЕ папки (не затирают друг друга).
- `freeze=3` (Swin) / grad-checkpointing уже включены → влезает в T4. Если всё же OOM — уменьши `EPOCHS` или `batch_p`.
- **Open-set + строгий A/B (zero-shot ↔ finetuned)** — Этап C, **локально** (НЕ здесь; `open_test` запечатан).


In [ ]:
# ════════ ПЕРЕКЛЮЧАТЕЛИ ════════
MODEL  = 'mega'    # 'mega' (MegaDescriptor-L-384, Swin) | 'miewid' (MiewID-msv3, EfficientNetV2)
EPOCHS = 15        # 15 — быстро увидеть сигнал; 30 — финальный прогон

# 1. GPU + Google Drive (всё состояние — в Drive, переживает сброс рантайма)
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_DIR = '/content/drive/MyDrive/triton_block4' + ('_miewid' if MODEL == 'miewid' else '')
os.makedirs(RUN_DIR + '/ckpt', exist_ok=True)
os.makedirs(RUN_DIR + '/oof', exist_ok=True)
print('MODEL:', MODEL, '| EPOCHS:', EPOCHS, '| RUN_DIR:', RUN_DIR)

In [ ]:
# 2. Зависимости. НЕ переустанавливаем torch (колабовская CUDA-сборка — переустановка ломает CUDA).
#    timm/albumentations — пин. Для MiewID transformers пинуется ДО первого импорта (рестарт не нужен).
!pip -q install timm==1.0.27 albumentations==2.0.8
if MODEL == 'miewid':
    !pip -q install "transformers==4.45.2"   # custom-код MiewID требует ~4.45 (новее → all_tied_weights_keys)
import torch, timm
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| timm', timm.__version__)
assert torch.cuda.is_available(), 'Нет GPU: Среда выполнения -> Сменить среду -> T4 GPU'

In [ ]:
# 3. Распаковка пакета + подключение src + верификация целостности (анти-битый-аплоад по sha256)
from pathlib import Path
import glob, json, hashlib, sys
_p = sorted(glob.glob('/content/drive/MyDrive/**/triton_block4_pack.zip', recursive=True))
assert _p, 'архив triton_block4_pack.zip не найден в MyDrive — залей его в любую папку Drive'
PACK = _p[0]; print('архив:', PACK)
!unzip -q -o "{PACK}" -d /content/work
ROOT = Path(sorted(glob.glob('/content/work/triton_block4_pack*'))[0])
%cd {ROOT}
sys.path.insert(0, str(ROOT / 'src'))           # пакет берётся из src/ (Colab Python может быть < requires-python)
!pip -q install -e . 2>/dev/null || echo 'pip install пропущен — пакет подключён через sys.path (это нормально)'
man = json.loads((ROOT / 'artifacts/embed/pack_manifest.json').read_text(encoding='utf-8'))
bad = [p for p, h in man['png'].items()
       if not (ROOT / p).exists() or hashlib.sha256((ROOT / p).read_bytes()).hexdigest() != h]
assert not bad, f'битые/отсутствуют PNG: {bad[:5]} (всего {len(bad)})'
import triton_crop, triton_data
print('pack OK:', man['n_png'], 'PNG ·', man['n_records'], 'записей · пакеты подключены')

In [ ]:
# 4. Конфиг под выбранную модель + готовый OOF-набор (фолды по особям посчитаны локально, seed=42)
import pandas as pd, numpy as np, torch
from PIL import Image
from dataclasses import replace
from triton_crop.config import load_embed_config

cfg = load_embed_config()
if MODEL == 'miewid':                                            # EfficientNetV2 51M, вход 440² целиком → batch меньше (T4)
    cfg = replace(cfg, base_model=cfg.miewid_model, image_size=cfg.miewid_image_size,
                  freeze_backbone_stages=0, batch_p=2, batch_k=3, epochs=EPOCHS)
else:                                                            # Swin-L 195M — морозим ~68% (layers[0..2]) + grad-checkpoint
    cfg = replace(cfg, freeze_backbone_stages=3, batch_p=4, batch_k=4, epochs=EPOCHS)
print(cfg.base_model, '| epochs', cfg.epochs, '| freeze', cfg.freeze_backbone_stages,
      '| PxK', cfg.batch_p, 'x', cfg.batch_k, '| folds', cfg.cross_fit_folds)

oof = pd.read_csv('artifacts/embed/oof_records.csv')
oof = oof.dropna(subset=['path_belly_oriented', 'path_unroll_ribbon']).reset_index(drop=True)
print('OOF:', len(oof), 'кадров ·', dict(oof.role.value_counts()), '· особь->1 fold:',
      bool((oof.groupby('individual_id').cross_fit_fold.nunique() == 1).all()))

def png_loader(path):
    return np.array(Image.open(ROOT / path).convert('RGB'))      # HWC uint8 — единый путь probe/gallery

def real_factory(name, device, num_classes=0):
    from triton_crop.embed_model import build_embedder
    e = build_embedder(name, device, num_classes=num_classes,
                       image_size=cfg.miewid_image_size, revision=(cfg.miewid_revision or None))
    m = e['model']
    if not hasattr(m, 'num_features'):                           # MiewID: dim головы ArcFace определяем прогоном (=2152)
        with torch.no_grad():
            _z = np.zeros((cfg.image_size, cfg.image_size, 3), 'uint8')
            m.num_features = int(m(e['transform'](Image.fromarray(_z)).unsqueeze(0).to(device)).shape[1])
    if hasattr(m, 'set_grad_checkpointing'):                     # экономия VRAM (T4): пересчёт активаций в backward
        m.set_grad_checkpointing(True)
    else:                                                        # MiewID: backbone спрятан внутри AutoModel-обёртки
        for attr in ('backbone', 'model', 'net', 'encoder'):
            bb = getattr(m, attr, None)
            if bb is not None and hasattr(bb, 'set_grad_checkpointing'):
                bb.set_grad_checkpointing(True)
                break
    return e

torch.cuda.empty_cache()

In [ ]:
# 5. Cross-fit finetune (ГЛАВНЫЙ шаг). Печатает прогресс по фолдам. resume=True: готовые fold{f}.pt не переучивает.
#    Анти-утечка гарантируется кодом: каждый кадр эмбеддится моделью, не видевшей его особь.
from triton_crop.embed_train import cross_fit_embed
result = cross_fit_embed(oof, cfg, build_embedder_fn=real_factory, image_loader=png_loader,
                         device='cuda', ckpt_dir=f'{RUN_DIR}/ckpt', epochs=cfg.epochs, resume=True)
print('OOF готов:', {v: result[v].shape for v in ('belly_oriented', 'unroll_ribbon')})

In [ ]:
# 6. Быстрая OOF-метрика (прокси «учится ли finetune») + выгрузка эмбеддингов/метрик в Drive
from triton_crop.ab_harness import per_probe_hits, summarize
role = result['role']; g = role == 'gallery'; p = role == 'probe'
ids = result['individual_id']; coh = result['cohort']
metrics = {}
for v in ('belly_oriented', 'unroll_ribbon'):
    h = per_probe_hits(result[v][p], ids[p], result[v][g], ids[g], ks=tuple(cfg.recall_ks))
    s = summarize(h, coh[p], ks=tuple(cfg.recall_ks))
    metrics[v] = s['overall']
    print(v, '-> OOF recall@1', round(s['overall']['recall@1'], 3), '@5', round(s['overall']['recall@5'], 3))
for v in ('belly_oriented', 'unroll_ribbon'):
    np.save(f'{RUN_DIR}/oof/{v}.npy', result[v])
for k in ('md5', 'role', 'individual_id', 'cohort'):
    np.save(f'{RUN_DIR}/oof/{k}.npy', result[k])
json.dump({'oof_recall': metrics, 'model': MODEL, 'base_model': cfg.base_model, 'epochs': cfg.epochs,
           'folds': cfg.cross_fit_folds, 'torch': torch.__version__, 'timm': timm.__version__},
          open(f'{RUN_DIR}/metrics.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('Выгружено в', RUN_DIR + '/oof + metrics.json (всё в Drive — переживёт сброс).')

## Этап C (локально, НЕ в этом ноутбуке)

Скачай `RUN_DIR/oof/*.npy` + `metrics.json` в `artifacts/embed/oof/`, затем локально:
- `python -m triton_crop.cli embed-ab --device mps` — строгий A/B: zero-shot ↔ finetuned,
  belly_oriented ↔ unroll_ribbon; McNemar/bootstrap; срезы overall + temporal + per-cohort.
- `python -m triton_crop.cli embed-eval-openset` — open-set AUROC known/new + порог Юдена на `open_dev`.

Для сравнения **MegaDescriptor ↔ MiewID**: прогони ноутбук дважды (`MODEL='mega'`, потом `'miewid'`),
скачай обе папки `oof`, сравни числа.

**Лицензия MiewID:** не объявлена на весах/коде (de facto all-rights-reserved); статья arXiv:2412.05602 —
CC-BY-4.0; NC-ограничений нет. Для академической ВКР — с цитированием; для прода — запросить у авторов модели (см. THIRD_PARTY_NOTICES.md).

**`test` / `open_test` НЕ вскрывать** — headline-KPI считается один раз в самом конце проекта.
